In [0]:
%pip install pdfplumber

In [0]:
import sys
from pathlib import Path

current = Path.cwd()

project_root = next(
    (
        path
        for path in (current, *current.parents)
        if (path / "src/orders").is_dir()
        and (path / "databricks.yml").is_file()
    ),
    None,
)

if project_root is None:
    raise RuntimeError("No se ha encontrado la raíz del proyecto.")

source_path = str(project_root / "src")

if source_path not in sys.path:
    sys.path.insert(0, source_path)

from orders.bronze_orders import ingest_orders

documents_path = (
    "abfss://receipts@graxianodatalake.dfs.core.windows.net/"
    "documents/"
)

In [0]:
result = ingest_orders(spark, dbutils, documents_path)
print(result)

# Los documentos válidos se conservan aunque otros fallen.
# Marcar el job como fallido permite recibir la alerta.
if result["errors"]:
    raise RuntimeError(
        f"{result['errors']} documentos no pudieron procesarse. "
        "Revisar pedidos.ops.document_processing."
    )

In [0]:
# display(spark.sql("""
#     SELECT
#         COUNT(*) AS filas,
#         COUNT(DISTINCT observation_id) AS ids_distintos
#     FROM pedidos.bronze.whatsapp_order_lines
# """))